In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import src.config as cfg
from src.io_paths import iteration_dir

VERSION = "V1"
out_dir = iteration_dir(VERSION)


In [ ]:
# Load trades and equity curve from iteration artifacts
trades_df = pd.read_csv(out_dir / "trades.csv", parse_dates=["entry_time","exit_time"])

# Load a sample of raw bars with indicators for the chart
# We'll use the first 5000 bars of training data for a manageable chart
import src.config as cfg
from src.data_loader import load_bars, split_train_test
from src.indicators import add_indicators
from src.strategy import generate_signals

df_raw = load_bars("data/NQ_1min.csv")
train, _ = split_train_test(df_raw, cfg.TRAIN_RATIO)
train = add_indicators(train, cfg)
train = generate_signals(train, cfg)

# Chart the first 5000 bars
CHART_BARS = 5000
chart_df = train.head(CHART_BARS).reset_index(drop=True)

# Filter trades that fall within the chart window
chart_start = chart_df["time"].min()
chart_end = chart_df["time"].max()
chart_trades = trades_df[(trades_df["entry_time"] >= chart_start) & (trades_df["entry_time"] <= chart_end)]
print(f"Chart window: {chart_start} to {chart_end}  |  Trades in window: {len(chart_trades)}")


In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    subplot_titles=["Price + EMA 8/21", "Z-score"],
    vertical_spacing=0.05,
)

# --- Row 1: Candlestick ---
fig.add_trace(go.Candlestick(
    x=chart_df["time"],
    open=chart_df["open"],
    high=chart_df["high"],
    low=chart_df["low"],
    close=chart_df["close"],
    name="Price",
    increasing_line_color="#26a69a",
    decreasing_line_color="#ef5350",
), row=1, col=1)

# EMA fast
fig.add_trace(go.Scatter(
    x=chart_df["time"], y=chart_df["ema_fast"],
    name=f"EMA {cfg.EMA_FAST}", line=dict(color="#1976D2", width=1),
), row=1, col=1)

# EMA slow
fig.add_trace(go.Scatter(
    x=chart_df["time"], y=chart_df["ema_slow"],
    name=f"EMA {cfg.EMA_SLOW}", line=dict(color="#FB8C00", width=1),
), row=1, col=1)

# Trade entry/exit markers
for _, trade in chart_trades.iterrows():
    color = "#26a69a" if trade["side"] == "long" else "#ef5350"
    symbol = "triangle-up" if trade["side"] == "long" else "triangle-down"
    fig.add_trace(go.Scatter(
        x=[trade["entry_time"]], y=[trade["entry_fill_price"]],
        mode="markers",
        marker=dict(symbol=symbol, size=8, color=color),
        name=f"{trade['side']} entry",
        showlegend=False,
    ), row=1, col=1)

# --- Row 2: Z-score ---
fig.add_trace(go.Scatter(
    x=chart_df["time"], y=chart_df["zscore"],
    name="Z-score", line=dict(color="#7B1FA2", width=1),
), row=2, col=1)

# Z-score band lines
for level, label in [(cfg.Z_BAND_K, f"+{cfg.Z_BAND_K}"), (-cfg.Z_BAND_K, f"-{cfg.Z_BAND_K}")]:
    fig.add_hline(y=level, line=dict(color="gray", dash="dash", width=1),
                  row=2, col=1, annotation_text=label, annotation_position="right")
fig.add_hline(y=0, line=dict(color="gray", width=0.5), row=2, col=1)

fig.update_layout(
    title=f"MNQ 1-min | EMA {cfg.EMA_FAST}/{cfg.EMA_SLOW} Crossover | {VERSION}",
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=True,
    height=700,
    template="plotly_dark",
)
fig.show()


In [ ]:
html_path = out_dir / "plotly_price_indicators.html"
fig.write_html(str(html_path))
print(f"Chart saved: {html_path}")
